In [ ]:
# ==========================================
# 1. MOUNT DRIVE & INSTALL DEPENDENCIES
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# Install Gradio AND the PyTorch Image Models (timm) library
!pip install gradio timm -q

import torch
import timm
import torchvision.transforms as transforms
import gradio as gr
from PIL import Image

# ==========================================
# 2. DEFINE CATEGORIES
# ==========================================
categories = [
    'battery', 'glass', 'metal', 'organic_waste',
    'paper_cardboard', 'plastic', 'textiles', 'trash'
]

# ==========================================
# 3. BUILD THE 'TIMM' SKELETON AND LOAD WEIGHTS
# ==========================================
print("Loading 'timm' EfficientNet-B0 architecture...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the skeleton using the timm library, setting the 8 categories immediately
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=len(categories))

print("Injecting your trained weights from Google Drive...")
model_path = '/content/drive/MyDrive/Aidataset/EfficientNet-B0_Model.pth'

# Load the weights into the perfectly matched timm skeleton
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval() # Set model to evaluation (testing) mode

# ==========================================
# 4. PREPROCESS THE IMAGE
# ==========================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 5. THE PREDICTION FUNCTION
# ==========================================
def classify_waste(image):
    # Convert Gradio image to PIL Image
    img = Image.fromarray(image).convert('RGB')

    # Apply mathematical transformations
    img_tensor = transform(img).unsqueeze(0).to(device)

    # Disable gradient tracking for faster inference
    with torch.no_grad():
        output = model(img_tensor)

        # Convert raw outputs to percentages using Softmax
        probabilities = torch.nn.functional.softmax(output[0], dim=0)

    # Create a dictionary mapping each category to its confidence score
    result = {categories[i]: float(probabilities[i]) for i in range(len(categories))}
    return result

# ==========================================
# 6. LAUNCH THE WEB INTERFACE
# ==========================================
print("Launching Web App...")
app = gr.Interface(
    fn=classify_waste,
    inputs=gr.Image(type="numpy", label="Upload a Photo of Waste"),
    outputs=gr.Label(num_top_classes=3, label="AI Predictions"),
    title="Intelligent Waste Sorter (EfficientNet-B0)",
    description="Upload a picture of trash, and the AI will analyze its spatial features to determine the correct recycling category.",
    allow_flagging="never"
)

# This generates a public URL you can open on your phone or laptop!
app.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading 'timm' EfficientNet-B0 architecture...
Injecting your trained weights from Google Drive...
Launching Web App...


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://973f9cefcdfb15e2a4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
